In [5]:
import logging
import os

# Suppress vLLM logs
logging.getLogger("vllm").setLevel(logging.ERROR)

# Suppress Hugging Face tokenizer warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# SPDX-License-Identifier: Apache-2.0

from typing import List
from pydantic import BaseModel, Field
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams

# Define structured output schema for marker genes
class MarkerGeneStrict(BaseModel):
    marker_gene_name: str = Field(
        ..., 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: str = Field(
        ..., 
        description="The name of the cell type for which this gene serves as a marker."
    )

class MarkerGeneListStrict(BaseModel):
    genes: List[MarkerGeneStrict]  # A list of marker genes extracted from the input text

json_schema = MarkerGeneListStrict.model_json_schema()

# Define system prompt and user prompt
system_prompt = """
You are an expert in genomics analyzing scientific literature to extract marker genes for different cell types. 

Your goal is to identify and structure marker gene data from the given text. For each marker gene mentioned, extract:
- The **gene name** (marker_gene_name).
- The **cell type** it is associated with (cell_type_name).

The data must be extracted as written in the text, without any modifications.

Return the results in **structured JSON format** matching this schema:
{
    "genes": [
        {
            "cell_type_name": "Neuron",
            "marker_gene_name": "GeneX"
        },
        ...
    ]
}
Ensure that the output is **valid JSON** and nothing else.
"""

user_prompt = """
After doublet removal and quality filtering, we considered a total of 197,721 cells (106,469 from PG and 91,252 from ING), identifying all cell types observed in human WAT (Fig. 2c, d, Supplementary Table 2) with the addition of 
distinct male and female epithelial populations (Dcdc2a+ and Erbb4+, respectively).
"""

# Combine system and user prompts
full_prompt = f"System: {system_prompt}\nUser: {user_prompt}"

# Define guided decoding (structured JSON output)
guided_decoding_params = GuidedDecodingParams(json=json_schema)

# Define sampling parameters (without speculative model)
sampling_params = SamplingParams(
    guided_decoding=guided_decoding_params,  # Enforce structured output
    temperature=0, 
    top_p=1,
)

# Initialize vLLM with speculative decoding
llm = LLM(
    model="meta-llama/Llama-3.2-1B",
    tensor_parallel_size=1,
    speculative_model="[ngram]",  # Enable speculative decoding
    num_speculative_tokens=5,
    ngram_prompt_lookup_max=4,
)

# Generate structured output with speculative decoding
outputs = llm.generate(prompts=full_prompt, sampling_params=sampling_params)

# Print extracted marker genes in structured JSON format
print(outputs[0].outputs[0].text)

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.55s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.55s/it]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.24s/it, est. speed input: 104.78 toks/s, output: 7.13 toks/s]

{ "genes": [ { "marker_gene_name": "GeneX", "


In [ ]:
# Define structured output schema for gene extraction
class GeneList(BaseModel):
    genes: List[str]  # A list of gene names extracted from the input text

json_schema = GeneList.model_json_schema()

# Define system prompt and user prompt
system_prompt = """
You are an expert in genomics analyzing scientific literature to extract gene names. 

Your goal is to identify and structure gene data from the given text. For each gene mentioned, extract:
- The **gene name** exactly as it appears in the text.

Only output the tokens corresponding to gene names, without any additional context.
"""

user_prompt = """
After doublet removal and quality filtering, we considered a total of 197,721 cells (106,469 from PG and 91,252 from ING), identifying all cell types observed in human WAT (Fig. 2c, d, Supplementary Table 2) with the addition of 
distinct male and female epithelial populations (Dcdc2a+ and Erbb4+, respectively).
"""

# Combine system and user prompts
full_prompt = f"System: {system_prompt}\nUser: {user_prompt}"

# Define guided decoding (structured JSON output)
guided_decoding_params = GuidedDecodingParams(json=json_schema)

# Define sampling parameters
sampling_params = SamplingParams(
    guided_decoding=guided_decoding_params,  # Enforce structured output
    temperature=0, 
    top_p=1,
)

# Initialize vLLM with speculative decoding
llm = LLM(
    model="meta-llama/Llama-3.2-1B",
    tensor_parallel_size=1,
    speculative_model="[ngram]",  # Enable speculative decoding
    num_speculative_tokens=3,
    ngram_prompt_lookup_max=4,
)

# Generate structured output with speculative decoding
outputs = llm.generate(prompts=full_prompt)#, sampling_params=sampling_params)

# Print extracted gene list in structured JSON format
print(outputs[0].outputs[0].text)

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:02<00:00,  2.16s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:02<00:00,  2.16s/it]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.76s/it, est. speed input: 87.95 toks/s, output: 9.08 toks/s]

We also identified 2,563 genes expressed in human WAT (Supplementary
